<a href="https://colab.research.google.com/github/vncwr/CropSight/blob/main/CropSight_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CropSight — Rice Disease Model Training

This notebook trains a MobileNetV2 model for rice leaf disease classification (3 classes: Bacterial Blight, Brown Spot, Leaf Smut), converts it to TFLite, and downloads the result.

Upload `Rice Leaf Diseases Dataset.zip` to the root of your Google Drive.



## Step 1 — Mount Drive & Extract Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

outer_zip = '/content/drive/MyDrive/Rice Leaf Diseases Dataset.zip'
if not os.path.exists(outer_zip):

    outer_zip = '/content/drive/MyDrive/Rice Lead Disease Dataset.zip'
    if not os.path.exists(outer_zip):
        raise FileNotFoundError('Upload Rice Leaf Diseases Dataset.zip to the ROOT of your Google Drive!')

print(f'Extracting {outer_zip}...')
with zipfile.ZipFile(outer_zip, 'r') as z:
    z.extractall('/content/dataset')

inner_zip = '/content/dataset/Rice Leaf Diseases Dataset/rice leaf diseases dataset.zip'
print(f'Extracting inner zip...')
with zipfile.ZipFile(inner_zip, 'r') as z:
    z.extractall('/content/dataset/extracted')

DATA_DIR = '/content/dataset/extracted/rice leaf diseases dataset'

for d in sorted(os.listdir(DATA_DIR)):
    full = os.path.join(DATA_DIR, d)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f'  {d}: {count} images')
print('Done!')

Mounted at /content/drive
Extracting /content/drive/MyDrive/Rice Leaf Diseases Dataset.zip...
Extracting inner zip...
  Bacterialblight: 1604 images
  Brownspot: 1620 images
  Leafsmut: 1460 images
Done!


## Step 2 — Split Dataset (80/10/10) + Augment Training Set

In [2]:
import shutil, random
from PIL import Image, ImageEnhance, ImageFilter
from collections import defaultdict

OUTPUT_DIR = '/content/splits'
SEED = 42

def augment_image(img):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    augmented = []
    augmented.append(img.rotate(30, fillcolor=(0,0,0)))
    augmented.append(img.rotate(-30, fillcolor=(0,0,0)))
    augmented.append(img.transpose(Image.FLIP_LEFT_RIGHT))
    augmented.append(ImageEnhance.Brightness(img).enhance(1.2))
    augmented.append(ImageEnhance.Brightness(img).enhance(0.8))
    augmented.append(ImageEnhance.Contrast(img).enhance(1.2))
    augmented.append(ImageEnhance.Contrast(img).enhance(0.8))
    augmented.append(img.filter(ImageFilter.GaussianBlur(radius=1)))
    return augmented

classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f'Classes: {classes}')

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for split in ['train', 'val', 'test']:
    for c in classes:
        os.makedirs(os.path.join(OUTPUT_DIR, split, c), exist_ok=True)

random.seed(SEED)
stats = defaultdict(lambda: defaultdict(int))

for c in classes:
    class_dir = os.path.join(DATA_DIR, c)
    images = sorted([f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    random.shuffle(images)
    n = len(images)
    n_test = max(1, int(n * 0.10))
    n_val = max(1, int(n * 0.10))
    test_imgs = images[:n_test]
    val_imgs = images[n_test:n_test+n_val]
    train_imgs = images[n_test+n_val:]

    for img_name in test_imgs:
        shutil.copy2(os.path.join(class_dir, img_name), os.path.join(OUTPUT_DIR, 'test', c, img_name))
        stats['test'][c] += 1
    for img_name in val_imgs:
        shutil.copy2(os.path.join(class_dir, img_name), os.path.join(OUTPUT_DIR, 'val', c, img_name))
        stats['val'][c] += 1
    for img_name in train_imgs:
        src = os.path.join(class_dir, img_name)
        shutil.copy2(src, os.path.join(OUTPUT_DIR, 'train', c, img_name))
        stats['train'][c] += 1
        try:
            with Image.open(src) as im:
                for j, aug in enumerate(augment_image(im)):
                    aug.save(os.path.join(OUTPUT_DIR, 'train', c, f'aug{j}_{img_name}'), 'JPEG', quality=95)
                    stats['aug'][c] += 1
        except Exception as e:
            pass

print('\n--- Split Summary ---')
print(f'{"Class":25s} | {"Train":>7s} | {"+Aug":>7s} | {"Val":>5s} | {"Test":>5s}')
for c in classes:
    print(f'{c:25s} | {stats["train"][c]:7d} | {stats["aug"][c]:7d} | {stats["val"][c]:5d} | {stats["test"][c]:5d}')
total_train = sum(stats['train'].values()) + sum(stats['aug'].values())
print(f'\nTotal training images: {total_train}')

Classes: ['Bacterialblight', 'Brownspot', 'Leafsmut']

--- Split Summary ---
Class                     |   Train |    +Aug |   Val |  Test
Bacterialblight           |    1284 |   10272 |   160 |   160
Brownspot                 |    1296 |   10368 |   162 |   162
Leafsmut                  |    1168 |    9344 |   146 |   146

Total training images: 33732


## Step 3 — Train MobileNetV2

In [3]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(OUTPUT_DIR, 'train'), image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True, seed=42)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(OUTPUT_DIR, 'val'), image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f'Classes ({num_classes}): {class_names}')

# Normalize to [-1, 1]
norm = tf.keras.layers.Rescaling(1./127.5, offset=-1)
train_ds = train_ds.map(lambda x, y: (norm(x), y)).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (norm(x), y)).prefetch(tf.data.AUTOTUNE)

base = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base.trainable = False
x = GlobalAveragePooling2D()(base.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(num_classes, activation='softmax')(x)
model = Model(inputs=base.input, outputs=out)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/best_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

# Phase 1: Train head
print('\n=== Phase 1: Training classification head ===')
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)

# Phase 2: Fine-tune top 30 layers
print('\n=== Phase 2: Fine-tuning top 30 layers ===')
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

with open('/content/class_names.txt', 'w') as f:
    for name in class_names:
        f.write(f'{name}\n')
print('\nTraining complete.')

TensorFlow 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 33732 files belonging to 3 classes.
Found 468 files belonging to 3 classes.
Classes (3): ['Bacterialblight', 'Brownspot', 'Leafsmut']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== Phase 1: Training classification head ===
Epoch 1/20
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.9211 - loss: 0.2157
Epoch 1: val_accuracy improved from None to 1.00000, saving model to /content/best_model.keras

Epoch 1: finished saving model to /content/best_model.keras
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 94s 71ms/step - accuracy: 0.9704 - loss: 0.0848 - val_accuracy: 1.0000 - val_loss: 0.0033 - learning_rate: 0.0010
Epoch 2/20
1054/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9956 - loss: 0.0153
Epoch 2: val_accuracy did not improve from 1.00000
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 53s 50ms/step - accuracy: 0.9960 - loss: 0.0133 - val_accuracy: 1.0000 - val_loss: 0.0019 - learning_rate: 0.0010

## Step 4 — Evaluate on Test Set

In [4]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(OUTPUT_DIR, 'test'), image_size=(224,224),
    batch_size=32, label_mode='categorical', shuffle=False)

test_ds_norm = test_ds.map(lambda x, y: (norm(x), y))

model = tf.keras.models.load_model('/content/best_model.keras')
predictions = model.predict(test_ds_norm)
y_pred = np.argmax(predictions, axis=1)
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])

print('\n=== Test Results ===')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred))

Found 468 files belonging to 3 classes.
15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 317ms/step

=== Test Results ===
                 precision    recall  f1-score   support

Bacterialblight     1.0000    1.0000    1.0000       160
      Brownspot     1.0000    1.0000    1.0000       162
       Leafsmut     1.0000    1.0000    1.0000       146

       accuracy                         1.0000       468
      macro avg     1.0000    1.0000    1.0000       468
   weighted avg     1.0000    1.0000    1.0000       468

Confusion Matrix:
[[160   0   0]
 [  0 162   0]
 [  0   0 146]]


## Step 5 — Convert to TFLite & Download

In [5]:
from google.colab import files

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('/content/model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite model size: {len(tflite_model)/1024/1024:.2f} MB')

# Save labels
with open('/content/labels.txt', 'w') as f:
    for name in class_names:
        f.write(f'{name}\n')

# Sanity check: compare float vs quantized
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()
out = interpreter.get_output_details()

matches = 0
total = 0
for images, _ in test_ds.take(50):
    img = norm(images[:1]).numpy().astype(np.float32)
    float_cls = np.argmax(model.predict(img, verbose=0))
    interpreter.set_tensor(inp[0]['index'], img)
    interpreter.invoke()
    tflite_cls = np.argmax(interpreter.get_tensor(out[0]['index']))
    if float_cls == tflite_cls:
        matches += 1
    total += 1

print(f'Float vs TFLite agreement: {matches}/{total} ({matches/total*100:.1f}%)')

print('\nDownloading model.tflite and labels.txt...')
files.download('/content/model.tflite')
files.download('/content/labels.txt')

Saved artifact at '/tmp/tmpd7siy_uc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132883145078160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145079888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145080080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145079312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145078736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145077968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145078352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145078544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145077392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145079504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132883145078928

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Float vs TFLite agreement: 15/15 (100.0%)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>